# Nomadic LLM Experiment — NB01: Setup & LoRA Expert Training

## 실험 설계 원칙

**비교 구조 (공정성 보장)**

모든 비교 모델은 **동일한 base model + 동일한 3개 LoRA expert**를 공유한다.  
차이는 오직 "어떻게 expert를 선택/전환하는가"의 라우팅 전략에만 있다.

```
Base Model (frozen)
     ├── LoRA_math   (math 도메인 학습)
     ├── LoRA_code   (code 도메인 학습)
     └── LoRA_lang   (language 도메인 학습)

라우팅 전략 비교:
  Vanilla         — base model, 라우팅 없음, T=0.7 고정
  DynamicTemp     — Δx → temperature 조절만, LoRA 없음  (신호 기여분 분리)
  StaticLoRA      — LoRA 있음, 라우팅 없음 (math LoRA 고정)  (expert 기여분 분리)
  FixedRouting    — LoRA 있음, 도메인 키워드 기반 고정 라우팅 (temporal 기여분 분리)
  Nomadic_NoDx    — PolicyNet 있음, Δx=0 고정 (Δx 기여분 분리)
  Nomadic_Full    — 전체 시스템
```

**절제 사유:**  
원래 12-Expert(math×4/code×4/lang×4)는 sub-expert 간 차별화 메커니즘이 없어
load balancing이 임의 분산될 뿐이다. **3-Expert**로 단순화하여 결과 해석을 명확히 한다.

**delta_err 재정의:**  
LLM autoregressive 생성에서 ground truth가 없으므로,  
`current_err = 1 - max_prob(logits)`로 정의한다 (model confidence proxy).
논문에서 이를 명확히 서술해야 한다.

**훈련/평가 분리:**  
- Train prompts: 도메인당 8개 (PolicyNet 지도학습용)
- Eval prompts: 도메인당 10개 (완전 분리, 학습에 사용 안 함)
- Stress prompts: 별도 세트 (시나리오 B~G용)

**파일 구조:**
```
NB01_setup_and_train.ipynb   — 이 파일: 설치, 모델 로드, LoRA 학습, PolicyNet 학습
NB02_baselines.ipynb         — Vanilla / DynamicTemp / StaticLoRA / FixedRouting 측정
NB03_nomadic_eval.ipynb      — Nomadic_NoDx / Nomadic_Full 측정
NB04_analysis.ipynb          — 집계, 표, 시각화, PAPER.md용 테이블 생성
```
결과는 모두 `{RUN_DIR}/` 아래 CSV로 저장되어 NB들 간에 공유된다.

In [ ]:
# ============================================================
# STEP 0: Google Drive 마운트 + 경로/모델 설정
# ============================================================
from google.colab import drive
import os, time
drive.mount('/content/drive', force_remount=True)

DRIVE_BASE   = '/content/drive/MyDrive/nomadic_llm_results'
MODELS_BASE  = '/content/drive/MyDrive/models'
RUN_ID       = time.strftime('%Y%m%d_%H%M%S')

# ── 모델 선택 (여기만 변경) ───────────────────────────────
MODEL_KEY = 'llama_8b'   # 'gemma4_26b' | 'mistral_24b' | 'llama_8b'

MODEL_REGISTRY = {
    'gemma4_26b':  os.path.join(MODELS_BASE, 'Gemma-4-26B-a4b-base'),
    'gemma4_E2B':  os.path.join(MODELS_BASE, 'Gemma-4-E2B-base'),
    'mistral_24b': os.path.join(MODELS_BASE, 'mistral-small-24b-base'),
    'llama_8b':    os.path.join(MODELS_BASE, 'Meta-Llama-3.1-8B-base'),
}
MODEL_PATH = MODEL_REGISTRY[MODEL_KEY]

# RUN_DIR: 이 RUN의 모든 결과가 저장되는 폴더 (NB 간 공유)
RUN_DIR = os.path.join(DRIVE_BASE, f'{MODEL_KEY}_{RUN_ID}')
os.makedirs(RUN_DIR, exist_ok=True)

# RUN_DIR 경로를 파일로 저장해 다른 NB에서 참조
with open(os.path.join(DRIVE_BASE, 'latest_run_dir.txt'), 'w') as f:
    f.write(RUN_DIR + '\n' + MODEL_KEY)

print(f'Model   : {MODEL_KEY}')
print(f'Path    : {MODEL_PATH}')
print(f'RUN_DIR : {RUN_DIR}')

In [ ]:
# ============================================================
# STEP 1: 패키지 설치
# ============================================================
!pip install -q transformers==4.44.0 accelerate bitsandbytes peft

import warnings
warnings.filterwarnings('ignore')

import math, json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import deque
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import (
    get_peft_model, LoraConfig, TaskType,
    PeftModel, prepare_model_for_kbit_training
)
from torch.optim import AdamW

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ============================================================
# STEP 2: 모델 로드 (4-bit NF4)
# ============================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f'{MODEL_KEY} 로드 중...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
base_model.eval()

# hidden_dim 확인
dummy = tokenizer('test', return_tensors='pt').input_ids.to(base_model.device)
with torch.no_grad():
    out = base_model(dummy, output_hidden_states=True)
HIDDEN_DIM = out.hidden_states[-1].shape[-1]
print(f'✅ 로드 완료 | hidden_dim={HIDDEN_DIM}')

# 설정값 저장 (다른 NB에서 로드용)
config = {'MODEL_KEY': MODEL_KEY, 'MODEL_PATH': MODEL_PATH,
          'RUN_DIR': RUN_DIR, 'HIDDEN_DIM': HIDDEN_DIM}
with open(os.path.join(RUN_DIR, 'run_config.json'), 'w') as f:
    json.dump(config, f, indent=2)
print('run_config.json 저장됨')

In [ ]:
# ============================================================
# STEP 3: Nomadic 컴포넌트 정의
# ============================================================

class HybridDeltaTracker:
    """
    Δx 신호 계산기.

    [LLM 적용 시 주의]
    합성 실험과 달리 LLM autoregressive 생성에서는 ground-truth가 없다.
    따라서 delta_err는 prediction error가 아니라
    "model confidence proxy" = 1 - max_prob(logits) 로 정의된다.
    이는 §2.2의 delta_err와 개념적으로 유사하지만 동일하지 않으므로
    논문에서 명확히 구분해야 한다.
    """
    def __init__(self, alpha=0.8, beta=0.85,
                 tau_min=2.0, tau_max=8.0,
                 tau_var_scale=6.0, tau_window=8):
        self.alpha = alpha
        self.beta  = beta
        self.tau_min = tau_min
        self.tau_max = tau_max
        self.tau_var_scale = tau_var_scale
        self.tau_window    = tau_window
        self.reset()

    def reset(self):
        self.prev_x       = None
        self.ema_err      = 0.0
        self.baseline_err = 0.0
        self.recent_de    = deque(maxlen=self.tau_window)
        self.history      = []

    def compute(self, current_x: torch.Tensor, current_err: float) -> dict:
        """
        current_x   : last-token hidden state (1, H)
        current_err : 1 - max_prob(logits)  (float, [0,1])
        """
        # ── delta_env: cosine distance between consecutive hidden states
        if self.prev_x is None:
            delta_env = 0.0
        else:
            cos = F.cosine_similarity(
                current_x.float().flatten(),
                self.prev_x.float().flatten(), dim=0)
            delta_env = float((1.0 - cos).clamp(0).item())
        self.prev_x = current_x.detach().clone()
        self.recent_de.append(delta_env)

        # ── delta_err: relative confidence degradation
        self.ema_err      = self.alpha * self.ema_err + (1 - self.alpha) * current_err
        self.baseline_err = self.beta  * self.baseline_err + (1 - self.beta) * current_err
        delta_err = max(0.0, self.ema_err - self.baseline_err)

        delta_hybrid = float(torch.tanh(
            torch.tensor(delta_env + delta_err)).item())

        sigma2   = float(np.var(self.recent_de)) if len(self.recent_de) >= 2 else 0.0
        tau_dyn  = self.tau_min + (self.tau_max - self.tau_min) / (
            1.0 + self.tau_var_scale * sigma2)

        rec = dict(delta_env=delta_env, delta_err=delta_err,
                   delta_hybrid=delta_hybrid, sigma2=sigma2, tau_dyn=tau_dyn)
        self.history.append(rec)
        return rec


class NomadicPolicyNet(nn.Module):
    """
    3-Expert 버전 PolicyNet.
    experts: ['math', 'code', 'language']

    설계 변경 (원래 12-Expert 대비):
    - 3-Expert로 단순화. 각 expert가 실질적으로 다른 도메인 데이터로 학습됨.
    - sub-expert(×4) 제거: 같은 도메인 데이터로 학습된 4개 복사본은
      load balancing의 arbitrary 분산만 유발함.
    """
    NUM_EXPERTS = 3
    EXPERT_KEYS = ['math', 'code', 'language']

    def __init__(self, hidden_dim: int, policy_hidden: int = 128):
        super().__init__()
        self.proj = nn.Linear(hidden_dim, policy_hidden)
        self.shared = nn.Sequential(
            nn.Linear(policy_hidden + 5, policy_hidden), nn.ReLU(),
            nn.Linear(policy_hidden, policy_hidden),     nn.ReLU(),
        )
        # head 1: stay(0) / switch(1)
        self.stay_switch_head = nn.Linear(policy_hidden, 2)
        # head 2: target expert (3)
        self.target_head      = nn.Linear(policy_hidden, self.NUM_EXPERTS)
        # head 3: soft(0) / hard(1) routing
        self.mode_head        = nn.Linear(policy_hidden, 2)

    def forward(self, hidden: torch.Tensor, meta_signals: torch.Tensor):
        if hidden.dim() == 3:
            hidden = hidden.squeeze(1)
        h   = F.relu(self.proj(hidden.float()))
        inp = torch.cat([h, meta_signals.float()], dim=-1)
        h   = self.shared(inp)
        return (
            F.softmax(self.stay_switch_head(h), dim=-1),
            F.softmax(self.target_head(h),      dim=-1),
            F.softmax(self.mode_head(h),        dim=-1),
        )


def build_meta_signals(rec: dict, device) -> torch.Tensor:
    """HybridDeltaTracker 출력 → PolicyNet 입력 5-dim 벡터"""
    return torch.tensor([[
        rec['delta_hybrid'],
        rec['delta_err'],
        rec['delta_hybrid'],                          # redundant channel (기존 호환)
        math.tanh(rec['sigma2'] * 10.0),
        math.tanh((rec['tau_dyn'] - 5.0) / 5.0),
    ]], dtype=torch.float32, device=device)


def topk_entropy(logits: torch.Tensor, k: int = 50) -> float:
    """Top-k entropy (엔트로피 근사, 생성 중 효율적 계산용)"""
    probs = F.softmax(logits, dim=-1)
    topk  = probs.topk(k).values
    topk  = topk / topk.sum()
    return float(-(topk * (topk + 1e-10).log()).sum().item())


EXPERT_KEYS = NomadicPolicyNet.EXPERT_KEYS
NUM_EXPERTS = NomadicPolicyNet.NUM_EXPERTS

print(f'✅ 컴포넌트 정의 완료 | experts={EXPERT_KEYS}')

In [ ]:
# ============================================================
# STEP 4: 프롬프트 정의 (훈련 / 평가 / 스트레스 완전 분리)
# ============================================================

# ──────────────────────────────────────────
# TRAIN: PolicyNet 지도학습용 (8개/도메인)
# → 이 프롬프트들은 평가에 절대 사용 안 함
# ──────────────────────────────────────────
TRAIN_PROMPTS = {
    'math': [
        "The derivative of x^3 is 3x^2. The derivative of x^4 is",
        "If a = 5 and b = 3, then a * b =",
        "log(1000) = 3이다. log(10000) =",
        "삼각형의 내각의 합은 180도이다. 오각형의 내각의 합은",
        "피타고라스 정리: 3^2 + 4^2 = 5^2이다. 5^2 + 12^2 =",
        "소수(prime) 목록: 2, 3, 5, 7, 11, 13. 다음 소수는",
        "1에서 100까지 합 = 5050. 1에서 10까지 합 =",
        "sin(0)=0, sin(π/2)=1, sin(π)=",
    ],
    'code': [
        "def factorial(n):\n    if n == 0: return 1\n    return n *",
        "arr = [3, 1, 4, 1, 5]\narr.sort()\nprint(arr)  #",
        "SELECT name FROM users WHERE age > 18 ORDER BY",
        "class Stack:\n    def __init__(self):\n        self.items =",
        "for i in range(5):\n    print(i * 2)  # outputs:",
        "import numpy as np\nmat = np.zeros((3, 3))\nmat[1][1] =",
        "function isPrime(n) {\n  for (let i = 2; i < n; i++)",
        "git add .\ngit commit -m 'fix'\ngit push origin",
    ],
    'language': [
        "처음에는 모든 것이 안정적이었다. 그러나 갑자기",
        "The ancient philosopher once said that the nature of reality is",
        "달빛이 창문을 통해 들어오던 그날 밤, 그는",
        "What if consciousness is not a product of the brain but",
        "In a world where time flows backwards,",
        "그 순간 그녀는 모든 것이 변했다는 것을 알았다. 왜냐하면",
        "The most beautiful thing about uncertainty is",
        "미래의 도시는 어떤 모습일까? 어쩌면",
    ],
}

# LoRA 학습용 (completion pair 형태)
LORA_TRAIN_PAIRS = {
    'math': [
        ("The integral of 2x is", " x^2 + C"),
        ("The square root of 256 is", " 16"),
        ("If f(x) = x^2, then f(3) =", " 9"),
        ("이차방정식 x^2 - 5x + 6 = 0의 근은", " x=2, x=3"),
        ("The area of a circle with radius 5 is", " 25π ≈ 78.54"),
        ("3 x 7 + 2 =", " 23"),
        ("cos(0) =", " 1"),
        ("The binomial coefficient C(5,2) =", " 10"),
        ("The derivative of sin(x) is", " cos(x)"),
        ("log base 2 of 8 =", " 3"),
    ],
    'code': [
        ("def add(a, b):\n    return", " a + b"),
        ("list(range(3)) ==", " [0, 1, 2]"),
        ("'hello'.upper() ==", " 'HELLO'"),
        ("[x**2 for x in range(4)] ==", " [0, 1, 4, 9]"),
        ("len([1,2,3,4,5]) ==", " 5"),
        ("max([3,1,4,1,5,9]) ==", " 9"),
        ("'abc'.replace('b','x') ==", " 'axc'"),
        ("sorted([3,1,2]) ==", " [1,2,3]"),
        ("isinstance(3, int) ==", " True"),
        ("type(3.14) ==", " <class 'float'>"),
    ],
    'language': [
        ("바람이 불어오는 곳,", " 그곳으로 가고 싶다"),
        ("The sky is painted in hues of", " orange and purple at sunset"),
        ("시간이 흐르면 흐를수록", " 그리움은 더 깊어진다"),
        ("In the quiet of the morning,", " thoughts drift like clouds"),
        ("그는 오래된 책을 펼쳐", " 먼지 쌓인 기억을 꺼냈다"),
        ("The universe whispers its secrets", " to those who listen carefully"),
        ("봄이 오면 꽃이 피듯", " 새로운 시작이 찾아온다"),
        ("Words can build bridges or", " burn them to the ground"),
        ("어둠이 짙을수록", " 별은 더 밝게 빛난다"),
        ("The essence of creativity lies", " in embracing the unknown"),
    ],
}

# ──────────────────────────────────────────
# EVAL: 학습과 완전 분리된 평가용 (10개/도메인)
# ──────────────────────────────────────────
EVAL_PROMPTS = {
    'math': [
        "24를 6으로 나누면",
        "The square root of 144 is",
        "If x + 5 = 12, then x =",
        "원의 넓이 공식은 πr²이다. 반지름이 3이면 넓이는",
        "등차수열 2, 5, 8, 11의 다음 항은",
        "The limit of (sin x)/x as x→0 is",
        "2^10 =",
        "행렬 [[1,0],[0,1]]의 행렬식(determinant)은",
        "The sum of angles in a hexagon is",
        "e^0 =",
    ],
    'code': [
        "# 피보나치 수열\ndef fib(n):\n    if n <= 1: return n\n    return",
        "const greeting = (name) => `Hello,",
        "try:\n    result = 10 / 0\nexcept ZeroDivisionError:",
        "<html><body><h1>Hello</h1></body></",
        "df = pd.DataFrame({'a': [1,2,3]})\ndf['b'] = df['a'] *",
        "from collections import Counter\nCounter('aabbcc') ==",
        "with open('file.txt', 'r') as f:\n    content =",
        "import re\nre.findall(r'\\d+', 'abc123def456') ==",
        "threading.Thread(target=worker).start()  #",
        "np.linspace(0, 1, 5) ==",
    ],
    'language': [
        "바람이 부는 날, 그는 오래된 편지를",
        "The paradox of freedom is that",
        "만약 내가 다시 태어난다면",
        "In the silence between words, there is",
        "존재한다는 것의 의미를 묻는다면",
        "고요한 새벽, 그녀는 창밖을 바라보며",
        "The road not taken reminds us that",
        "기억이란 결국",
        "When the last star fades,",
        "변하지 않는 것들에 대한 믿음은",
    ],
}

# ──────────────────────────────────────────
# STRESS: 시나리오 B~G 전용 (평가와도 분리)
# ──────────────────────────────────────────
# 시나리오 B: 도메인 급변 시퀀스 (math → language)
SCENARIO_B_SEQUENCE = [
    # --- math phase (앞 5개)
    "The quadratic formula for ax^2 + bx + c = 0 gives x =",
    "The determinant of [[a,b],[c,d]] is",
    "Euler's number e ≈",
    "The fundamental theorem of calculus states that",
    "A geometric series with ratio r converges if",
    # --- language phase (뒤 5개)
    "그 긴 여정의 끝에서 그는 비로소",
    "The feeling of standing at the edge of something vast is",
    "시간은 기억 속에서만",
    "What remains when everything has been forgotten is",
    "어쩌면 우리는 모두",
]
SCENARIO_B_SHIFT_POINT = 5  # 인덱스 5부터 language로 전환

# 시나리오 C: 노이즈 주입 (eval math 프롬프트 기반)
def inject_noise(text, noise_rate=0.15):
    """문자 수준 노이즈 주입 (오타 시뮬레이션)"""
    import random
    chars = list(text)
    n_noise = max(1, int(len(chars) * noise_rate))
    for _ in range(n_noise):
        idx = random.randint(0, len(chars) - 1)
        chars[idx] = random.choice('abcdefghijklmnopqrstuvwxyz!@#$%')
    return ''.join(chars)

SCENARIO_C_CLEAN  = EVAL_PROMPTS['math'][:5]  # eval에서 가져오되 stress 세트로 분류
SCENARIO_C_NOISY  = [inject_noise(p) for p in SCENARIO_C_CLEAN]

# 시나리오 D: 도메인 중첩 (hybrid prompts)
SCENARIO_D_HYBRID = [
    "수학적으로 증명하면: if f(x) is a beautiful function,",
    "def love(x):\n    # returns the feeling of",
    "The algorithm of grief follows a O(n) complexity where n is",
    "시간의 미분은 dt이며, 그것이 흐르면",
    "SELECT meaning FROM life WHERE purpose IS NOT",
]

# 시나리오 E: 유혹-회복 (math 맥락 중 language 문장 주입)
SCENARIO_E_CASES = [
    # (math_prefix, lure_injection, math_continuation)
    ("The integral of x^2 is x^3/3 + C. Therefore",
     " but wait — the moon tonight reminds me of lost summers.",
     " Returning to calculus: the integral of x^3 is"),
    ("Solving 2x + 3 = 11 gives x = 4. Similarly",
     " though somehow sadness lingers in these symbols.",
     " solving 3x + 2 = 14 gives x ="),
    ("The Pythagorean theorem: a^2 + b^2 = c^2. Thus",
     " yet this equation feels like the distance between hearts.",
     " for a=3, b=4, c ="),
]

# 시나리오 F: 정보 밀도 (dense = 고밀도, sparse = 저밀도)
SCENARIO_F_DENSE = [
    "The Riemann hypothesis states that all non-trivial zeros of ζ(s) lie on",
    "Gödel's incompleteness theorem implies that any consistent formal system",
    "The P vs NP problem asks whether every problem whose solution can be verified",
]
SCENARIO_F_SPARSE = [
    "The number 2 is",
    "A simple word is",
    "One plus one equals",
]

# 시나리오 G: 장기 고착도 (600 토큰 단일 도메인)
SCENARIO_G_PROMPTS = {
    'math':     "Let us systematically enumerate the properties of prime numbers. "
                "First, 2 is the only even prime. Second, all primes greater than 2 are odd. "
                "Third, the prime counting function π(x) grows as x/ln(x). The next key property is",
    'code':     "# Comprehensive Python data structures tutorial\n"
                "# List: ordered, mutable. Tuple: ordered, immutable. Dict: key-value, mutable.\n"
                "# Set: unordered, unique elements.\n"
                "# Usage example:\ndata = {'name': 'Alice', 'scores': [95, 87, 92]}\nprint(data[",
    'language': "인간의 기억이란 무엇인가. 우리는 과거를 기억하지만 그 기억은 늘 왜곡된다. "
                "철학자들은 말했다: 기억은 현재의 산물이라고. 그렇다면 우리가 기억하는 나는 "
                "진짜 나인가, 아니면 현재의 내가 만들어낸 허상인가. 이 질문의 끝에 다다르면",
}

# ── 저장 (다른 NB에서 로드용) ────────────────────────────────
prompts_data = {
    'TRAIN_PROMPTS': TRAIN_PROMPTS,
    'LORA_TRAIN_PAIRS': LORA_TRAIN_PAIRS,
    'EVAL_PROMPTS': EVAL_PROMPTS,
    'SCENARIO_B_SEQUENCE': SCENARIO_B_SEQUENCE,
    'SCENARIO_B_SHIFT_POINT': SCENARIO_B_SHIFT_POINT,
    'SCENARIO_C_CLEAN': SCENARIO_C_CLEAN,
    'SCENARIO_C_NOISY': SCENARIO_C_NOISY,
    'SCENARIO_D_HYBRID': SCENARIO_D_HYBRID,
    'SCENARIO_E_CASES': SCENARIO_E_CASES,
    'SCENARIO_F_DENSE': SCENARIO_F_DENSE,
    'SCENARIO_F_SPARSE': SCENARIO_F_SPARSE,
    'SCENARIO_G_PROMPTS': SCENARIO_G_PROMPTS,
}
with open(os.path.join(RUN_DIR, 'prompts.json'), 'w', encoding='utf-8') as f:
    json.dump(prompts_data, f, ensure_ascii=False, indent=2)

print('✅ 프롬프트 정의 완료')
for d in ['math', 'code', 'language']:
    print(f'  {d}: train={len(TRAIN_PROMPTS[d])}, eval={len(EVAL_PROMPTS[d])}, '
          f'lora_pairs={len(LORA_TRAIN_PAIRS[d])}')

In [ ]:
# ============================================================
# STEP 5: LoRA Expert 학습 (3개: math / code / language)
# ============================================================

LORA_EPOCHS    = 10
LORA_LR        = 5e-4
MAX_SEQ_LEN    = 128

def get_lora_config():
    return LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=['q_proj', 'v_proj'],
        bias='none',
    )


def encode_pair(prompt: str, completion: str) -> dict:
    """(prompt, completion) 쌍을 토큰화. completion 부분만 label 활성화."""
    full     = prompt + completion
    enc_full = tokenizer(full,   add_special_tokens=True,
                         max_length=MAX_SEQ_LEN, truncation=True)
    enc_pmt  = tokenizer(prompt, add_special_tokens=True,
                         max_length=MAX_SEQ_LEN, truncation=True)
    input_ids = enc_full['input_ids']
    labels    = [-100] * len(enc_pmt['input_ids']) + \
                input_ids[len(enc_pmt['input_ids']):]
    # 길이 맞추기
    labels = labels[:len(input_ids)]
    return {
        'input_ids': torch.tensor(input_ids, dtype=torch.long),
        'labels':    torch.tensor(labels,    dtype=torch.long),
    }


def train_lora_expert(domain: str, pairs: list, save_dir: str) -> str:
    """
    domain: 'math' | 'code' | 'language'
    pairs : [(prompt, completion), ...]
    """
    save_path = os.path.join(save_dir, f'lora_{domain}')
    if os.path.exists(save_path):
        print(f'  {domain}: 이미 존재, 스킵 ({save_path})')
        return save_path

    print(f'  {domain} LoRA 학습 중...')
    dataset = [encode_pair(p, c) for p, c in pairs]

    model_prep = prepare_model_for_kbit_training(base_model)
    lora_model = get_peft_model(model_prep, get_lora_config())
    lora_model.train()
    lora_model.print_trainable_parameters()

    optimizer = AdamW(lora_model.parameters(), lr=LORA_LR)
    losses = []

    for epoch in range(LORA_EPOCHS):
        epoch_loss = 0.0
        for sample in dataset:
            input_ids = sample['input_ids'].unsqueeze(0).to(base_model.device)
            labels    = sample['labels'].unsqueeze(0).to(base_model.device)
            optimizer.zero_grad()
            out  = lora_model(input_ids=input_ids, labels=labels)
            loss = out.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(lora_model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
        avg = epoch_loss / len(dataset)
        losses.append(avg)
        if (epoch + 1) % 2 == 0:
            print(f'    epoch {epoch+1:2d}/{LORA_EPOCHS} | loss={avg:.4f}')

    lora_model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)

    # 손실 곡선 저장
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(losses)
    ax.set_title(f'LoRA {domain} Training Loss')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.grid(alpha=0.3)
    fig.savefig(os.path.join(save_dir, f'lora_{domain}_loss.png'),
                dpi=120, bbox_inches='tight')
    plt.close(fig)

    print(f'  ✅ {domain} 저장 완료: {save_path}')
    return save_path


print('LoRA Expert 학습 시작...')
expert_paths = {}
for domain in EXPERT_KEYS:
    expert_paths[domain] = train_lora_expert(
        domain, LORA_TRAIN_PAIRS[domain], RUN_DIR
    )

# 경로 저장
with open(os.path.join(RUN_DIR, 'expert_paths.json'), 'w') as f:
    json.dump(expert_paths, f, indent=2)

print('\n✅ 전체 LoRA 학습 완료')
for k, v in expert_paths.items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# STEP 6: PolicyNet 학습 데이터 수집
# ============================================================
# 수집 방식: TRAIN_PROMPTS로 각 도메인 expert를 실행하고
# hidden state + meta signal + 정답 레이블을 기록
#
# 레이블 정의:
#   math/code   → stay(0), hard routing(1)  : 예측 가능, 안정적
#   language    → switch(1), soft routing(0) : 탐색적, 전환 허용
# ============================================================

STEPS_PER_PROMPT = 15  # 프롬프트당 수집 스텝 수

def _load_expert(key):
    m = PeftModel.from_pretrained(base_model, expert_paths[key])
    m.eval()
    return m

def _unload_expert(m):
    del m
    torch.cuda.empty_cache()


# domain별 정답 레이블
DOMAIN_LABELS = {
    'math':     {'switch': 0, 'mode': 1},  # stay, hard
    'code':     {'switch': 0, 'mode': 1},  # stay, hard
    'language': {'switch': 1, 'mode': 0},  # switch, soft
}

def collect_training_data():
    train_data = []
    for domain in EXPERT_KEYS:
        labels = DOMAIN_LABELS[domain]
        expert = _load_expert(domain)
        tracker = HybridDeltaTracker()

        for prompt in TRAIN_PROMPTS[domain]:
            input_ids = tokenizer(
                prompt, return_tensors='pt'
            ).input_ids.to(base_model.device)
            tracker.reset()

            for _ in range(STEPS_PER_PROMPT):
                with torch.no_grad():
                    out    = expert(input_ids, output_hidden_states=True)
                logits = out.logits[:, -1, :]
                hidden = out.hidden_states[-1][:, -1, :]
                err    = float(1.0 - F.softmax(logits, dim=-1).max().item())
                rec    = tracker.compute(hidden, err)
                meta   = build_meta_signals(rec, base_model.device)

                # target expert 인덱스: 같은 도메인 유지
                tgt_idx = EXPERT_KEYS.index(domain)

                train_data.append({
                    'hidden':       hidden.cpu(),
                    'meta':         meta.cpu(),
                    'switch_label': labels['switch'],
                    'target_label': tgt_idx,
                    'mode_label':   labels['mode'],
                    'domain':       domain,
                })

                # 다음 스텝: greedy 토큰 추가
                tok = logits.argmax(-1, keepdim=True)
                input_ids = torch.cat([input_ids, tok], dim=-1)
                if tok.item() == tokenizer.eos_token_id:
                    break

        _unload_expert(expert)
        print(f'  {domain}: {len([d for d in train_data if d["domain"]==domain])} samples')

    return train_data


print('PolicyNet 학습 데이터 수집 중...')
train_data = collect_training_data()
print(f'✅ 총 {len(train_data)}개 샘플 수집')

In [ ]:
# ============================================================
# STEP 7: PolicyNet 학습
# ============================================================
POLICY_EPOCHS = 30
POLICY_LR     = 1e-3
POLICY_BATCH  = 32

policy_net = NomadicPolicyNet(HIDDEN_DIM, policy_hidden=128)
policy_net = policy_net.to(base_model.device)
print(f'PolicyNet 파라미터: {sum(p.numel() for p in policy_net.parameters()):,}')

optimizer_p = AdamW(policy_net.parameters(), lr=POLICY_LR)
ce_loss     = nn.CrossEntropyLoss()

losses_p = []
policy_net.train()

for epoch in range(POLICY_EPOCHS):
    indices    = torch.randperm(len(train_data))
    epoch_loss = 0.0
    n_batches  = 0

    for start in range(0, len(train_data), POLICY_BATCH):
        batch = [train_data[i] for i in indices[start:start + POLICY_BATCH]]

        h_batch = torch.cat([d['hidden'] for d in batch]).to(base_model.device)
        m_batch = torch.cat([d['meta']   for d in batch]).to(base_model.device)
        sw_lbl  = torch.tensor([d['switch_label'] for d in batch],
                                dtype=torch.long, device=base_model.device)
        tg_lbl  = torch.tensor([d['target_label'] for d in batch],
                                dtype=torch.long, device=base_model.device)
        md_lbl  = torch.tensor([d['mode_label']   for d in batch],
                                dtype=torch.long, device=base_model.device)

        optimizer_p.zero_grad()
        ss, tgt, mode = policy_net(h_batch, m_batch)

        loss = (ce_loss(ss,   sw_lbl)
              + ce_loss(tgt,  tg_lbl)
              + ce_loss(mode, md_lbl))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy_net.parameters(), 1.0)
        optimizer_p.step()
        epoch_loss += loss.item()
        n_batches  += 1

    avg = epoch_loss / n_batches
    losses_p.append(avg)
    if (epoch + 1) % 5 == 0:
        print(f'  epoch {epoch+1:2d}/{POLICY_EPOCHS} | loss={avg:.4f}')

# ── 학습 완료 후 switch saturation 체크 ─────────────────────
# 문제: switch_prob가 모든 샘플에서 1.0으로 수렴하면
# stay/switch 구분이 무너진 것 → 경고 출력
policy_net.eval()
all_h = torch.cat([d['hidden'] for d in train_data]).to(base_model.device)
all_m = torch.cat([d['meta']   for d in train_data]).to(base_model.device)
all_d = [d['domain'] for d in train_data]

with torch.no_grad():
    ss_all, _, _ = policy_net(all_h, all_m)
switch_probs = ss_all[:, 1].cpu().numpy()

for domain in EXPERT_KEYS:
    mask = [i for i, d in enumerate(all_d) if d == domain]
    sp   = switch_probs[mask].mean()
    flag = ' ⚠️ SATURATED' if sp > 0.90 or sp < 0.10 else ''
    print(f'  {domain:10s}: mean switch_prob = {sp:.3f}{flag}')

# 저장
policy_path = os.path.join(RUN_DIR, 'policy_net.pt')
torch.save(policy_net.state_dict(), policy_path)

# 손실 곡선
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(losses_p)
ax.set_title('PolicyNet Training Loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.grid(alpha=0.3)
fig.savefig(os.path.join(RUN_DIR, 'policy_net_loss.png'), dpi=120, bbox_inches='tight')
plt.close(fig)

print(f'\n✅ PolicyNet 저장: {policy_path}')

In [ ]:
# ============================================================
# STEP 8: 학습 전/후 PolicyNet 엔트로피 비교 (Table 4 §4.5 재현)
# ============================================================
# 이 셀은 학습이 제대로 됐는지 sanity check.
# EVAL 프롬프트를 사용 (TRAIN이 아님)

MAX_STEPS_CHECK = 15

def measure_entropy_untrained(domain, prompts, n_steps=MAX_STEPS_CHECK):
    """PolicyNet 없이 base entropy 측정 (Vanilla 기준)"""
    entropies = []
    for prompt in prompts:
        input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(base_model.device)
        for _ in range(n_steps):
            with torch.no_grad():
                out = base_model(input_ids)
            logits = out.logits[:, -1, :]
            entropies.append(topk_entropy(logits))
            tok = logits.argmax(-1, keepdim=True)
            input_ids = torch.cat([input_ids, tok], dim=-1)
            if tok.item() == tokenizer.eos_token_id: break
    return float(np.mean(entropies))


def measure_entropy_with_policy(domain, prompts, n_steps=MAX_STEPS_CHECK):
    """PolicyNet 포함 entropy 측정"""
    expert = _load_expert(domain)
    tracker = HybridDeltaTracker()
    entropies, switch_probs = [], []

    for prompt in prompts:
        input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(base_model.device)
        tracker.reset()
        for _ in range(n_steps):
            with torch.no_grad():
                out = expert(input_ids, output_hidden_states=True)
            logits = out.logits[:, -1, :]
            hidden = out.hidden_states[-1][:, -1, :]
            err    = float(1.0 - F.softmax(logits, dim=-1).max().item())
            rec    = tracker.compute(hidden, err)
            meta   = build_meta_signals(rec, base_model.device)

            with torch.no_grad():
                ss, _, _ = policy_net(hidden.unsqueeze(0), meta)
            d = rec['delta_hybrid']
            temp = 0.35 + (0.90 - 0.35) * d
            entropies.append(topk_entropy(logits / max(temp, 1e-4)))
            switch_probs.append(ss[0, 1].item())
            tok = logits.argmax(-1, keepdim=True)
            input_ids = torch.cat([input_ids, tok], dim=-1)
            if tok.item() == tokenizer.eos_token_id: break

    _unload_expert(expert)
    return float(np.mean(entropies)), float(np.mean(switch_probs))


print('엔트로피 비교 측정 중 (EVAL 프롬프트 사용)...')

# stable 도메인 = math + code, transition = language
stable_prompts = EVAL_PROMPTS['math'][:5] + EVAL_PROMPTS['code'][:5]
trans_prompts  = EVAL_PROMPTS['language'][:5]

stable_H_raw = measure_entropy_untrained('math', stable_prompts)
trans_H_raw  = measure_entropy_untrained('language', trans_prompts)

stable_H_policy, stable_sw = measure_entropy_with_policy('math', stable_prompts)
trans_H_policy,  trans_sw  = measure_entropy_with_policy('language', trans_prompts)

print('\n' + '='*60)
print(f'PolicyNet Sanity Check — {MODEL_KEY}')
print('='*60)
print(f'{"Setting":35s} | Stable H | Trans H |  ΔH')
print('-'*60)
print(f'{MODEL_KEY+" — untrained":35s} | {stable_H_raw:.3f}    | {trans_H_raw:.3f}   | {trans_H_raw-stable_H_raw:+.3f}')
print(f'{MODEL_KEY+" — trained":35s} | {stable_H_policy:.3f}    | {trans_H_policy:.3f}   | {trans_H_policy-stable_H_policy:+.3f}')
print(f'\nSwitch prob — stable: {stable_sw:.3f}, transition: {trans_sw:.3f}')

if abs(stable_sw - trans_sw) < 0.1:
    print('⚠️  경고: stable/transition 간 switch_prob 차이가 작음 (<0.1).')
    print('   PolicyNet이 stay/switch를 구분하지 못하고 있을 수 있음.')
    print('   이 경우 NB03의 Nomadic_Full 결과 해석에 주의 필요.')
else:
    print('✅ PolicyNet이 stable/transition을 정상 구분 중')

# 체크 결과 저장
sanity = {
    'stable_H_raw': stable_H_raw, 'trans_H_raw': trans_H_raw,
    'stable_H_policy': stable_H_policy, 'trans_H_policy': trans_H_policy,
    'stable_sw': stable_sw, 'trans_sw': trans_sw,
}
with open(os.path.join(RUN_DIR, 'sanity_check.json'), 'w') as f:
    json.dump(sanity, f, indent=2)

print(f'\n✅ NB01 완료. 다음: NB02_baselines.ipynb')